# 10 — Saving, Loading & Deployment

Checkpointing, `state_dict` vs whole-model save, TorchScript, ONNX export, mixed precision, and inference best practices.

## state_dict: the recommended save format

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))

torch.save(model.state_dict(), "model_weights.pt")

model2 = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))  # must recreate the architecture
model2.load_state_dict(torch.load("model_weights.pt", weights_only=True))
model2.eval()


## Saving the Whole Model (less portable, avoid for sharing)

In [ ]:
torch.save(model, "full_model.pt")          # pickles the entire object, including class definition
loaded = torch.load("full_model.pt", weights_only=False)   # requires original class importable


## Full Training Checkpoint (resume training later)

In [ ]:
checkpoint = {
    "epoch": 10,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": torch.optim.Adam(model.parameters()).state_dict(),
    "loss": 0.023,
}
torch.save(checkpoint, "checkpoint.pt")

ckpt = torch.load("checkpoint.pt", weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
optimizer = torch.optim.Adam(model.parameters())
optimizer.load_state_dict(ckpt["optimizer_state_dict"])
start_epoch = ckpt["epoch"]


## Saving Across Devices

In [ ]:
# save on GPU, load on CPU
model.load_state_dict(torch.load("model_weights.pt", map_location=torch.device("cpu")))

# save on CPU, load on GPU
device = torch.device("cuda")
model.to(device)
model.load_state_dict(torch.load("model_weights.pt", map_location=device))


## TorchScript: tracing (for models with static control flow)

In [ ]:
model.eval()
example_input = torch.randn(1, 10)
traced = torch.jit.trace(model, example_input)
traced.save("model_traced.pt")

loaded_traced = torch.jit.load("model_traced.pt")
loaded_traced(example_input)


## TorchScript: scripting (for models with data-dependent control flow, e.g. if/loops)

In [ ]:
class ControlFlowModel(nn.Module):
    def forward(self, x):
        if x.sum() > 0:
            return x * 2
        return x * -1

scripted = torch.jit.script(ControlFlowModel())
scripted.save("model_scripted.pt")


## ONNX Export (interoperability with other runtimes: ONNX Runtime, TensorRT, etc.)

In [ ]:
model.eval()
dummy_input = torch.randn(1, 10)

torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},  # variable batch size
    opset_version=17,
)


In [ ]:
# running inference with ONNX Runtime
# pip install onnxruntime
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("model.onnx")
outputs = session.run(None, {"input": dummy_input.numpy()})
outputs[0].shape


## Mixed Precision Training (torch.amp)

Uses float16/bfloat16 for most ops, keeping master weights in float32 — speeds up training on modern GPUs with minimal accuracy loss.

In [ ]:
from torch.amp import autocast, GradScaler

device = torch.device("cuda")
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scaler = GradScaler(device="cuda")

for xb, yb in [(torch.randn(8, 10), torch.randint(0, 2, (8,)))]:
    xb, yb = xb.to(device), yb.to(device)
    optimizer.zero_grad()
    with autocast(device_type="cuda"):
        preds = model(xb)
        loss = nn.CrossEntropyLoss()(preds, yb)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()


## Mixed Precision / Fast Inference

In [ ]:
model.eval()
with torch.no_grad(), autocast(device_type="cuda"):
    output = model(xb)


## Quantization (smaller, faster models for deployment)

In [ ]:
# dynamic quantization — easiest, good for LSTMs/Linear-heavy models on CPU
quantized_model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)
torch.save(quantized_model.state_dict(), "quantized_model.pt")


## Inference Best Practices

In [ ]:
model.eval()                  # disable dropout, use running BatchNorm stats
with torch.no_grad():          # disable autograd tracking, saves memory/compute
    output = model(xb)

# for repeated single-sample inference, batch requests when possible for throughput


## Common Gotchas
- Always prefer `state_dict` over whole-model pickling — pickled models break if the class definition moves or changes.
- `weights_only=True` (default changing across PyTorch versions) should be used unless you trust the checkpoint source — safer against arbitrary code execution in pickled files.
- Forgetting `model.eval()` before export/inference leaves Dropout/BatchNorm in training mode, producing inconsistent outputs.
- ONNX export requires static or explicitly declared dynamic shapes — implicit reliance on Python control flow (e.g. `if` on tensor values) won't translate correctly unless scripted.
- Mixed precision needs `GradScaler` to prevent gradient underflow in float16 — skipping it can silently produce NaN losses.